#### 260113 Tutorial of Pseudo Bulk Generation


In [1]:
BASE_DIR = '/workspace/HDDX/TopicModel_Deconv'

import pandas as pd
import scanpy as sc

import sys
sys.path.append(f'{BASE_DIR}/github/omics-util')
import simulation

***
##### 1. single cellのデータを読み込む

In [2]:
adata = sc.read_h5ad(f'{BASE_DIR}/datasource/scRNASeq/LiverCellAtlas/mouseStStAll/processed/liver_adata_148202x19052.h5ad')
adata

AnnData object with n_obs × n_vars = 148202 × 19052
    obs: 'cell_type', 'n_genes', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo', 'n_counts'
    var: 'gene_name', 'n_cells', 'mt', 'ribo'

***
##### 2. 対象とする細胞種について、ランダムに比率を割り当てる
- 事前分布は、ディリクレ分布と一様分布を選択可能（methodの変数）
- 訓練データとして、8000サンプル作成する例

In [ ]:
dat = simulation.LiverCellAtlas_Simulator(sample_size=8000, method='dirichlet')
dat.assign()
summary_df = dat.summary_df
summary_df.to_csv(f'{BASE_DIR}/github/omics-util/simulation/data/gs_8000x12.csv')

summary_df.head()

,Neutrophils,Monocytes & Monocyte-derived cells,Kupffer cells,NK cells,T cells,B cells,pDCs,cDC1s,cDC2s,Hepatocytes,Cholangiocytes,Fibroblasts
0,0.025960,0.166518,0.072841,0.050503,0.009384,0.009382,0.003310,0.111260,0.050843,0.048921,0.313807,0.137272
1,0.074255,0.001254,0.211295,0.107737,0.014395,0.012103,0.012219,0.021877,0.044865,0.364546,0.067733,0.067721
2,0.054612,0.033241,0.091388,0.014508,0.033365,0.044061,0.058803,0.148514,0.021509,0.010006,0.336309,0.153685
3,0.033263,0.041347,0.002191,0.043090,0.008613,0.003099,0.136995,0.155281,0.076121,0.129452,0.002187,0.368361
4,0.031604,0.008941,0.100283,0.050465,0.011323,0.059465,0.003044,0.208824,0.026051,0.401301,0.053618,0.045080


- リークしないように、trainデータ制作用の細胞とtestデータ制作用とに分ける

In [4]:
dat.set_data(summary_df=summary_df, cell_idx_dict=None, adata=adata)
dat.split_cell_idx(save_dir=f'{BASE_DIR}/github/omics-util/simulation/data/cell_idx', train_ratio=0.7)
cell_idx_dict = dat.cell_idx_dict

Neutrophils: 3350 cells detected
Monocytes & Monocyte-derived cells: 18907 cells detected
Kupffer cells: 30687 cells detected
NK cells: 1857 cells detected
T cells: 9482 cells detected
B cells: 5443 cells detected
pDCs: 5158 cells detected
cDC1s: 3745 cells detected
cDC2s: 4355 cells detected
Hepatocytes: 19196 cells detected
Cholangiocytes: 809 cells detected
Fibroblasts: 4511 cells detected


```
- B_test_idx.pkl
- B_train_idx.pkl
...
のように、trainとtestを作成するための、各細胞のindex情報を格納したpklファイルが作成される。
```
- testにreal world dataを使用する場合は、trainのみ作成すれば良く、リークも考慮しなくて良いのでtrain_ratio=1.0にしているする方が細胞数も稼げて良い。

In [5]:
cell_idx_dict = dat.cell_idx_dict
pd.to_pickle(cell_idx_dict, f'{BASE_DIR}/github/omics-util/simulation/data/cell_idx/cell_idx_dict.pkl')

***
##### 3. 割り振った比率に基づいて疑似Bulkを生成する
- mode='train'を指定して、train用のindexから作成する
- リークしないテストデータを作成したい場合は、summary_dfを更新したうえで、modeを切り替える

In [9]:
# create train (80 samples) dataset
summary_df = pd.read_csv(f'{BASE_DIR}/github/omics-util/simulation/data/gs_8000x12.csv', index_col=0)
cell_idx_dict = pd.read_pickle(f'{BASE_DIR}/github/omics-util/simulation/data/cell_idx/cell_idx_dict.pkl')

summary_df = summary_df.iloc[0:10,:] # 時短で10サンプルの作成

dat = simulation.LiverCellAtlas_Simulator(sample_size=10, method='dirichlet')  
dat.set_data(summary_df=summary_df, cell_idx_dict=cell_idx_dict, adata=adata)
bulk_df = dat.create_sim_bulk(pool_size=500, mode='train')
bulk_df.to_csv(f'{BASE_DIR}/github/omics-util/simulation/data/bulk_19052_10.csv')

bulk_df.head()

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [02:09<00:00, 12.99s/it]


,0,1,2,3,4,5,6,7,8,9
Xkr4,46.0,27.0,42.0,63.0,3.0,47.0,42.0,37.0,26.0,29.0
Rp1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
Sox17,1.0,2.0,0.0,3.0,1.0,3.0,1.0,1.0,3.0,1.0
Mrpl15,137.0,202.0,130.0,161.0,284.0,145.0,158.0,162.0,157.0,232.0
Lypla1,78.0,97.0,84.0,94.0,129.0,63.0,121.0,106.0,78.0,111.0
